[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 词嵌入（word2vec）

这个 notebook 是[《动手学深度学习》](http://www.d2l.ai/index.html)一书中 [word2vec 的 mxnet 实现](http://www.d2l.ai/chapter_natural-language-processing-pretraining/word2vec.html) 的 PyTorch 改编版。

**更新（2023）** 现在[预训练 word2vec](http://www.d2l.ai/chapter_natural-language-processing-pretraining/word2vec-pretraining.html#pretraining-word2vec) 一节提供了 PyTorch 实现，和这里给出的略有不同。

自然语言是一个我们用来表达意义的复杂系统。在这个系统中，词是语言意义的基本单元。顾名思义，词向量是用来表示词的向量，也可以看作词的特征向量。把词映射到实数向量的技术也叫词嵌入。在过去几年里，词嵌入逐渐成为自然语言处理的基础知识。

## 为什么不使用 one-hot 向量？


虽然 one-hot 词向量很容易构造，但它们通常不是一个好的选择。其中一个主要原因是，one-hot 词向量无法准确表达不同词之间的相似度，比如我们常用的余弦相似度。对于向量 $\boldsymbol{x}, \boldsymbol{y} \in \mathbb{R}^d$，它们的余弦相似度是它们之间夹角的余弦：

$$\frac{\boldsymbol{x}^\top \boldsymbol{y}}{\|\boldsymbol{x}\| \|\boldsymbol{y}\|} \in [-1, 1].$$

由于任意两个不同词的 one-hot 向量之间的余弦相似度为 0，很难用 one-hot 向量来准确表示多个不同词之间的相似度。

[Word2vec](https://code.google.com/archive/p/word2vec/) 就是解决上述问题的工具。它为每个词表示成一个定长向量，并用这些向量更好地表示不同词之间的相似度和类比关系。Word2vec 工具包含两种模型：跳字模型（[skip-gram](Distributed representations of words and phrases and their compositionality.)）和连续词袋模型（[CBOW](Efficient estimation of word representations in vector space)）。接下来，我们来看一下这两种模型及其训练方法。


## 跳字模型（Skip-Gram）

跳字模型假设一个词可以用来生成文本序列中它周围的词。例如，假设文本序列是 "the"、"man"、"loves"、"his" 和 "son"。我们用 "loves" 作为中心目标词，并把上下文窗口大小设为 2。如下所示，给定中心目标词 "loves"，跳字模型关心生成上下文词 "the"、"man"、"his" 和 "son"（这些词与中心词的距离不超过 2 个词）的条件概率，也就是

$$\mathbb{P}(\textrm{"the"},\textrm{"man"},\textrm{"his"},\textrm{"son"}\mid\textrm{"loves"}).$$

我们假设，给定中心目标词后，上下文词是彼此独立生成的。在这种情况下，上面的公式可以改写为

$$\mathbb{P}(\textrm{"the"}\mid\textrm{"loves"})\cdot\mathbb{P}(\textrm{"man"}\mid\textrm{"loves"})\cdot\mathbb{P}(\textrm{"his"}\mid\textrm{"loves"})\cdot\mathbb{P}(\textrm{"son"}\mid\textrm{"loves"}).$$

![跳字模型关心给定中心目标词时生成上下文词的条件概率。](https://www.di.ens.fr/~lelarge/skip-gram.svg)


在跳字模型中，每个词用两个 $d$ 维向量表示，用来计算条件概率。我们假设词在词典中的索引为 $i$，当它作为中心目标词时，向量表示为 $\boldsymbol{v}_i\in\mathbb{R}^d$；当它作为上下文词时，向量表示为 $\boldsymbol{u}_i\in\mathbb{R}^d$。设中心目标词 $w_c$ 和上下文词 $w_o$ 在词典中的索引分别为 $c$ 和 $o$。给定中心目标词生成上下文词的条件概率，可以通过对向量内积做 softmax 运算得到：

$$\mathbb{P}(w_o \mid w_c) = \frac{\text{exp}(\boldsymbol{u}_o^\top \boldsymbol{v}_c)}{ \sum_{i \in \mathcal{V}} \text{exp}(\boldsymbol{u}_i^\top \boldsymbol{v}_c)},$$

其中词表索引集合为 $\mathcal{V} = \{0, 1, \ldots, |\mathcal{V}|-1\}$。假设给定一个长度为 $T$ 的文本序列，其中时间步 $t$ 的词记为 $w^{(t)}$。假设给定中心词后上下文词独立生成。当上下文窗口大小为 $m$ 时，跳字模型的似然函数是给定任意中心词生成所有上下文词的联合概率

$$ \prod_{t=1}^{T} \prod_{-m \leq j \leq m,\ j \neq 0} \mathbb{P}(w^{(t+j)} \mid w^{(t)}),$$

训练结束后，对于词典中索引为 $i$ 的任意词，我们将得到它的两个词向量 $\boldsymbol{v}_i$ 和 $\boldsymbol{u}_i$。在自然语言处理（NLP）应用中，一般使用跳字模型中的中心目标词向量作为词的表征向量。

## 连续词袋模型（CBOW）

连续词袋模型（CBOW）与跳字模型类似。最大的区别在于，CBOW 模型假设中心目标词是基于文本序列中它前后的上下文词生成的。还是用同样的文本序列 "the"、"man"、"loves"、"his" 和 "son"，其中 "loves" 是中心目标词，给定上下文窗口大小为 2，CBOW 模型关心基于上下文词 "the"、"man"、"his" 和 "son" 生成目标词 "loves" 的条件概率（如下所示），也就是

$$\mathbb{P}(\textrm{"loves"}\mid\textrm{"the"},\textrm{"man"},\textrm{"his"},\textrm{"son"}).$$

![CBOW 模型关心从给定的上下文词生成中心目标词的条件概率。](https://www.di.ens.fr/~lelarge/cbow.svg)

由于 CBOW 模型中有多个上下文词，我们将它们的词向量取平均，然后使用与跳字模型相同的方法计算条件概率。我们假设 $\boldsymbol{v_i}\in\mathbb{R}^d$ 和 $\boldsymbol{u_i}\in\mathbb{R}^d$ 分别是词典中索引为 $i$ 的词的上下文词向量和中心目标词向量（注意与跳字模型的符号相反）。设中心目标词 $w_c$ 在词典中的索引为 $c$，上下文词 $w_{o_1}, \ldots, w_{o_{2m}}$ 在词典中的索引为 $o_1, \ldots, o_{2m}$。这样，从给定上下文词生成中心目标词的条件概率是

$$\mathbb{P}(w_c \mid w_{o_1}, \ldots, w_{o_{2m}}) = \frac{\text{exp}\left(\frac{1}{2m}\boldsymbol{u}_c^\top (\boldsymbol{v}_{o_1} + \ldots + \boldsymbol{v}_{o_{2m}}) \right)}{ \sum_{i \in \mathcal{V}} \text{exp}\left(\frac{1}{2m}\boldsymbol{u}_i^\top (\boldsymbol{v}_{o_1} + \ldots + \boldsymbol{v}_{o_{2m}}) \right)}.$$


为简洁起见，记 $\mathcal{W}_o= \{w_{o_1}, \ldots, w_{o_{2m}}\}$，$\bar{\boldsymbol{v}}_o = \left(\boldsymbol{v}_{o_1} + \ldots + \boldsymbol{v}_{o_{2m}} \right)/(2m)$。上面的公式可以简化为

$$\mathbb{P}(w_c \mid \mathcal{W}_o) = \frac{\exp\left(\boldsymbol{u}_c^\top \bar{\boldsymbol{v}}_o\right)}{\sum_{i \in \mathcal{V}} \exp\left(\boldsymbol{u}_i^\top \bar{\boldsymbol{v}}_o\right)}.$$

给定一个长度为 $T$ 的文本序列，我们假设时间步 $t$ 的词是 $w^{(t)}$，上下文窗口大小为 $m$。CBOW 模型的似然函数就是从上下文词生成任意中心目标词的概率。

$$ \prod_{t=1}^{T}  \mathbb{P}(w^{(t)} \mid  w^{(t-m)}, \ldots,  w^{(t-1)},  w^{(t+1)}, \ldots,  w^{(t+m)}).$$

与跳字模型不同，在 CBOW 模型中我们通常使用上下文词向量作为词的表征向量。


# 近似训练

跳字模型的核心特征是使用 softmax 运算来计算基于给定中心目标词 $w_c$ 生成上下文词 $w_o$ 的条件概率。

$$\mathbb{P}(w_o \mid w_c) = \frac{\text{exp}(\boldsymbol{u}_o^\top \boldsymbol{v}_c)}{ \sum_{i \in \mathcal{V}} \text{exp}(\boldsymbol{u}_i^\top \boldsymbol{v}_c)}.$$

该条件概率对应的对数损失为

$$-\log \mathbb{P}(w_o \mid w_c) =
-\boldsymbol{u}_o^\top \boldsymbol{v}_c + \log\left(\sum_{i \in \mathcal{V}} \text{exp}(\boldsymbol{u}_i^\top \boldsymbol{v}_c)\right).$$


由于 softmax 运算考虑了上下文词可以是词典 $\mathcal{V}$ 中的任意词，上面提到的损失实际上包含了词典大小项数的求和。因此，每一步的梯度计算都包含词典大小项数的求和。对于含有数十万甚至数百万个词的较大词典，计算每个梯度的开销可能过高。为了降低这种计算复杂度，我们将在本节介绍一种近似训练方法：负采样。由于跳字模型和 CBOW 模型之间没有本质区别，本节只用跳字模型作为例子来介绍这两种训练方法。



## 负采样

负采样修改了原来的目标函数。给定中心目标词 $w_c$ 的一个上下文窗口，我们把上下文词 $w_o$ 出现在上下文窗口中看作一个事件，并按下面的公式计算这个事件的概率

$$\mathbb{P}(D=1\mid w_c, w_o) = \sigma(\boldsymbol{u}_o^\top \boldsymbol{v}_c),$$

这里的 $\sigma$ 函数与 sigmoid 激活函数的定义相同：

$$\sigma(x) = \frac{1}{1+\exp(-x)}.$$

我们首先考虑通过最大化文本序列中所有事件的联合概率来训练词向量。给定一个长度为 $T$ 的文本序列，我们假设时间步 $t$ 的词是 $w^{(t)}$，上下文窗口大小为 $m$。现在考虑最大化联合概率

$$ \prod_{t=1}^{T} \prod_{-m \leq j \leq m,\ j \neq 0} \mathbb{P}(D=1\mid w^{(t)}, w^{(t+j)}).$$

然而，模型中包含的事件只考虑了正样本。在这种情况下，只有当所有词向量都相等且取值趋近于无穷大时，上面的联合概率才能最大化到 1。显然，这样的词向量毫无意义。负采样通过额外采样负例来让目标函数更有意义。假设当上下文词 $w_o$ 出现在中心目标词 $w_c$ 的上下文窗口中时事件 $P$ 发生，我们按分布 $\mathbb{P}(w)$ 采样 $K$ 个没有出现在上下文窗口中的词作为噪声词。我们假设噪声词 $w_k$（$k=1, \ldots, K$）不出现在中心目标词 $w_c$ 的上下文窗口中的事件是 $N_k$。假设正负样本的事件 $P$ 和 $N_1, \ldots, N_K$ 彼此独立。通过负采样，我们可以把上面只考虑正样本的联合概率改写为


$$ \prod_{t=1}^{T} \prod_{-m \leq j \leq m,\ j \neq 0} \mathbb{P}(w^{(t+j)} \mid w^{(t)}),$$

这里条件概率被近似为
$$ \mathbb{P}(w^{(t+j)} \mid w^{(t)}) =\mathbb{P}(D=1\mid w^{(t)}, w^{(t+j)})\prod_{k=1,\ w_k \sim \mathbb{P}(w)}^K \mathbb{P}(D=0\mid w^{(t)}, w_k).$$


设时间步 $t$ 的词 $w^{(t)}$ 在文本序列中的索引为 $i_t$，噪声词 $w_k$ 在词典中的索引为 $h_k$。上述条件概率的对数损失是

$$
\begin{aligned}
-\log\mathbb{P}(w^{(t+j)} \mid w^{(t)})
=& -\log\mathbb{P}(D=1\mid w^{(t)}, w^{(t+j)}) - \sum_{k=1,\ w_k \sim \mathbb{P}(w)}^K \log\mathbb{P}(D=0\mid w^{(t)}, w_k)\\
=&-  \log\, \sigma\left(\boldsymbol{u}_{i_{t+j}}^\top \boldsymbol{v}_{i_t}\right) - \sum_{k=1,\ w_k \sim \mathbb{P}(w)}^K \log\left(1-\sigma\left(\boldsymbol{u}_{h_k}^\top \boldsymbol{v}_{i_t}\right)\right)\\
=&-  \log\, \sigma\left(\boldsymbol{u}_{i_{t+j}}^\top \boldsymbol{v}_{i_t}\right) - \sum_{k=1,\ w_k \sim \mathbb{P}(w)}^K \log\sigma\left(-\boldsymbol{u}_{h_k}^\top \boldsymbol{v}_{i_t}\right).
\end{aligned}
$$

这里，训练中每一步的梯度计算不再与词典大小相关，而是与 $K$ 线性相关。当 $K$ 取一个较小的常数时，负采样每一步的计算开销更低。

更多细节见 [On word embeddings - Part 2: Approximating the Softmax](https://www.ruder.io/word-embeddings-softmax/)


# Word2vec 的实现




In [1]:
import collections

import math
import numpy as np

import random
import sys
import time
import zipfile

## 处理数据集

PTB（Penn Tree Bank）是一个常用的小型[语料](https://github.com/tomsercu/lstm/tree/master/data)。它从《华尔街日报》的文章中取样，包含训练集、验证集和测试集。我们将在 PTB 训练集上训练词嵌入模型。数据集的每一行相当于一个句子。句子中的所有词用空格分隔。


In [2]:
## # # colab 环境配置
!mkdir data
%cd data
!wget https://raw.githubusercontent.com/tomsercu/lstm/master/data/ptb.train.txt
ROOT_DIR='content'

/home/mlelarge/courses/dataflowr/notebooks/Module8/data
--2023-05-15 10:56:05--  https://raw.githubusercontent.com/tomsercu/lstm/master/data/ptb.train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5101618 (4.9M) [text/plain]
Saving to: 'ptb.train.txt'

ptb.train.txt       100%[===================>]   4.87M  19.6MB/s    in 0.2s    

2023-05-15 10:56:07 (19.6 MB/s) - 'ptb.train.txt' saved [5101618/5101618]



In [3]:
import os
from pathlib import Path

ROOT_DIR = Path.home()

In [4]:
data_path = os.path.join(ROOT_DIR,'data/')
file = 'ptb.train.txt'
with open(data_path+file, 'r') as f:
    lines = f.readlines()
    # 循环里 st 是 sentence 的缩写
    raw_dataset = [st.split() for st in lines]

'# sentences: %d' % len(raw_dataset)

'# sentences: 42068'

对于数据集中的前三个句子，打印每个句子的词数以及前五个词。这个数据集的结束字符是 "&lt;eos&gt;"，不常见的词都用 "&lt;unk&gt;" 表示，数字被替换成 "N"。


In [5]:
for st in raw_dataset[:3]:
    print(len(st),st[:5])

24 ['aer', 'banknote', 'berlitz', 'calloway', 'centrust']
15 ['pierre', '<unk>', 'N', 'years', 'old']
11 ['mr.', '<unk>', 'is', 'chairman', 'of']


### 建立词索引

为了简单起见，我们只保留数据集中至少出现 5 次的词。


In [6]:
# 循环里 tk 是 token 的缩写
counter = collections.Counter([tk for st in raw_dataset for tk in st])
counter = dict(filter(lambda x: x[1] >= 5, counter.items()))

In [7]:
counter['the']

50770

In [8]:
idx_to_token = [tk for tk, _ in counter.items()]
token_to_idx = {tk: idx for idx, tk in enumerate(idx_to_token)}
dataset = [[token_to_idx[tk] for tk in st if tk in token_to_idx]
           for st in raw_dataset]
num_tokens = sum([len(st) for st in dataset])
'# tokens: %d' % num_tokens

'# tokens: 887100'

In [9]:
idx_to_token[:5]

['pierre', '<unk>', 'N', 'years', 'old']

In [10]:
token_to_idx['consensus']

4827

In [11]:
token_to_idx['pierre']

0

In [12]:
dataset[:2]

[[], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 2]]

### 二次采样

在文本数据中，一般会有些词出现频率很高，比如英语里的 "the"、"a" 和 "in"。一般来说，在上下文窗口里，一个词（比如 "chip"）与一个低频词（比如 "microprocessor"）同时出现，比它与一个高频词（比如 "the"）同时出现，对训练词嵌入模型更好。因此，在训练词嵌入模型时，我们可以对词进行二次采样[2]。具体来说，数据集中的每个被索引的词 $w_i$ 都会以一定概率被丢弃。丢弃概率为：

$$ \mathbb{P}(w_i) = \max\left(1 - \sqrt{\frac{t}{f(w_i)}}, 0\right),$$

这里 $f(w_i)$ 是词 $w_i$ 的实例数占数据集中总词数的比例，常数 $t$ 是一个超参数（本实验中设为 $10^{-4}$）。可以看到，只有当 $f(w_i) > t$ 时，词 $w_i$ 才可能在二次采样中被丢弃。词的频率越高，它的丢弃概率越大。


In [13]:
def discard(idx):
    return random.uniform(0, 1) < 1 - math.sqrt(
        1e-4 / counter[idx_to_token[idx]] * num_tokens)

subsampled_dataset = [[tk for tk in st if not discard(tk)] for st in dataset]
'# tokens: %d' % sum([len(st) for st in subsampled_dataset])

'# tokens: 376109'

In [14]:
def compare_counts(token):
    return '# %s: before=%d, after=%d' % (token, sum(
        [st.count(token_to_idx[token]) for st in dataset]), sum(
        [st.count(token_to_idx[token]) for st in subsampled_dataset]))

compare_counts('the')

'# the: before=50770, after=2138'

In [15]:
compare_counts('join')

'# join: before=45, after=45'

### 提取中心目标词和上下文词

我们把与中心目标词距离不超过上下文窗口大小的词，作为给定中心目标词的下下文词。下面的定义函数提取所有的中心目标词及其上下文词。它会在整数 1 和 `max_window_size`（最大上下文窗口）之间均匀随机地采样一个整数，作为上下文窗口大小。


In [16]:
def get_centers_and_contexts(dataset, max_window_size):
    centers, contexts = [], []
    for st in dataset:
        # 每个句子至少需要 2 个词才能构成一个
        # "中心目标词 - 上下文词"对
        if len(st) < 2:
            continue
        centers += st
        for center_i in range(len(st)):
            window_size = random.randint(1, max_window_size)
            indices = list(range(max(0, center_i - window_size),
                                 min(len(st), center_i + 1 + window_size)))
            # 把中心目标词从上下文词中排除
            indices.remove(center_i)
            contexts.append([st[idx] for idx in indices])
    return centers, contexts

In [17]:
tiny_dataset = [list(range(7)), list(range(7, 10))]
print('dataset', tiny_dataset)
for center, context in zip(*get_centers_and_contexts(tiny_dataset, 2)):
    print('center', center, 'has contexts', context)

dataset [[0, 1, 2, 3, 4, 5, 6], [7, 8, 9]]
center 0 has contexts [1, 2]
center 1 has contexts [0, 2, 3]
center 2 has contexts [1, 3]
center 3 has contexts [2, 4]
center 4 has contexts [3, 5]
center 5 has contexts [4, 6]
center 6 has contexts [4, 5]
center 7 has contexts [8, 9]
center 8 has contexts [7, 9]
center 9 has contexts [8]


实验中，我们把最大上下文窗口大小设为 5。下面提取数据集中所有的中心目标词及其上下文词。


In [18]:
all_centers, all_contexts = get_centers_and_contexts(subsampled_dataset, 5)

## 负采样

我们使用负采样进行近似训练。对于一对中心词和上下文词，我们随机采样 $K$ 个噪声词（实验中 $K=5$）。根据 Word2vec 论文的建议，噪声词采样概率 $\mathbb{P}(w)$ 是词 $w$ 的词频占全部词频之比再取 0.75 次方。


In [19]:
def get_negatives(all_contexts, sampling_weights, K):
    all_negatives, neg_candidates, i = [], [], 0
    population = list(range(len(sampling_weights)))
    for contexts in all_contexts:
        negatives = []
        while len(negatives) < len(contexts) * K:
            if i == len(neg_candidates):
                # 根据每个词的权重（sampling_weights）随机生成 k 个词的索引作为噪声词。为了
                # 高效计算，k 可以设得稍大一点
                # 
                i, neg_candidates = 0, random.choices(population, sampling_weights, k=int(1e5))
            neg, i = neg_candidates[i], i + 1
            # 噪声词不能是上下文词
            if neg not in set(contexts):
                negatives.append(neg)
        all_negatives.append(negatives)
    return all_negatives

sampling_weights = [counter[w]**0.75 for w in idx_to_token]
all_negatives = get_negatives(all_contexts, sampling_weights, 5)

In [20]:
all_negatives[0]

[391, 1042, 355, 885, 23, 10, 566, 1589, 718, 2880, 705, 6806, 178, 325, 7]

In [21]:
all_contexts[0]

[6, 8, 11]

## 读取数据

我们从数据集中提取了所有的中心目标词 `all_centers`，以及每个中心目标词的上下文词 `all_contexts` 和噪声词 `all_negatives`。我们将以随机小批次的方式读取它们。

在一个小批次数据中，第 $i$ 个样本包含一个中心词及其对应的 $n_i$ 个上下文词和 $m_i$ 个噪声词。由于每个样本的上下文窗口大小可能不同，上下文词和噪声词的数量之和 $n_i+m_i$ 也会不同。构造小批次时，我们把每个样本的上下文词和噪声词拼接起来，并补 0 直到所有拼接的长度一致，也就是所有拼接的长度为 $\max_i n_i+m_i$（`max_len`）。为了避免填充对损失函数计算的影响，我们构造掩码变量 `masks`，它的每个元素对应上下文词和噪声词拼接 `contexts_negatives` 中的一个元素。当 `contexts_negatives` 变量中的某个元素是填充时，掩码变量 `masks` 中相同位置的元素为 0，否则取值为 1。为了区分正样本和负样本，我们还需要在 `contexts_negatives` 变量中区分上下文词和噪声词。基于掩码变量的构造，我们只需要创建一个与 `contexts_negatives` 变量形状相同的标签变量 `labels`，并把对应上下文词（正样本）的元素设为 1，其余设为 0。

接下来，我们将实现小批次读取函数 `batchify`。它的小批次输入 `data` 是一个列表，长度为 batch size，每个元素包含中心目标词 `center`、上下文词 `context` 和噪声词 `negative`。这个函数返回的小批次数据符合我们需要的格式，例如它包含掩码变量。我们把这个函数包装进一个 `Dataset` 模块里。


In [22]:
import torch
import torch.nn as nn

In [23]:
from torch.utils.data import Dataset, DataLoader

In [24]:
class PTB_dataset(Dataset):
    
    def __init__(self, all_centers, all_contexts, all_negatives):
        self.all_centers, self.all_contexts_negatives, self.all_masks, self.all_labels = self.batchify(list(zip(all_centers,all_contexts,all_negatives)))
        
    def __len__(self):
        return len(self.all_centers)
    
    def __getitem__(self,idx):
        return self.all_centers[idx], self.all_contexts_negatives[idx], self.all_masks[idx], self.all_labels[idx]
        
    def batchify(self,data):
        max_len = max(len(c) + len(n) for _, c, n in data)
        centers, contexts_negatives, masks, labels = [], [], [], []
        for center, context, negative in data:
            cur_len = len(context) + len(negative)
            centers += [center]
            contexts_negatives += [context + negative + [0] * (max_len - cur_len)]
            masks += [[1] * cur_len + [0] * (max_len - cur_len)]
            labels += [[1] * len(context) + [0] * (max_len - len(context))]
        return (torch.tensor(centers).view((-1, 1)), torch.tensor(np.array(contexts_negatives)),
            torch.tensor(np.array(masks)), torch.tensor(np.array(labels)))
        

In [25]:
ptbdata = PTB_dataset(all_centers, all_contexts, all_negatives)

In [26]:
ptbdata[1]

(tensor([6]),
 tensor([   0,    8,   11,   12,   13, 2334, 7310,   39, 5792, 4815, 4608,  474,
         1190,  143, 1978,   83, 4337, 4476,  220,   94, 7261, 6052,  457,   68,
          464, 3739, 1145, 1476, 1139, 1219,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

下面就是我们的 dataloader。


In [27]:
batch_size = 512

data_loader = DataLoader(ptbdata, batch_size, shuffle=True,
                              num_workers=4)
for batch in data_loader:
    for name, data in zip(['centers', 'contexts_negatives', 'masks',
                           'labels'], batch):
        print(name, 'shape:', data.shape)
    break

centers shape: torch.Size([512, 1])
contexts_negatives shape: torch.Size([512, 60])
masks shape: torch.Size([512, 60])
labels shape: torch.Size([512, 60])


## 跳字模型

你将使用 embedding 层和小批次乘法 [`torch.einsum`](https://pytorch.org/docs/stable/torch.html#torch.bmm) 来实现跳字模型。这些方法也常用于实现其他自然语言处理应用。


In [28]:
# 取自 spotlight 库，见
# https://github.com/maciejkula/spotlight/blob/master/spotlight/layers.py
class ScaledEmbedding(nn.Embedding):
    """
    Embedding layer that initialises its values
    to using a normal variable scaled by the inverse
    of the emedding dimension.
    """
    def reset_parameters(self):
        """
        Initialize parameters.
        """
        self.weight.data.normal_(0, 1.0 / self.embedding_dim)
        if self.padding_idx is not None:
            self.weight.data[self.padding_idx].fill_(0)

### 跳字模型前向计算

在前向计算中，跳字模型的输入包含中心目标词索引 `center` 以及拼接后的上下文词和噪声词索引 `contexts_and_negatives`。其中，`center` 变量的形状为（batch size, 1），而 `contexts_and_negatives` 变量的形状为（batch size, `max_len`）。这两个变量先通过词嵌入层从词索引变换成词向量，然后通过小批次乘法得到形状为（batch size, 1, `max_len`）的输出。输出中的每个元素是中心目标词向量与上下文词向量或噪声词向量的内积。


In [29]:
class Skip_gram(nn.Module):
    def __init__(self, input_dim, embed_size = 100):
        super(Skip_gram, self).__init__()
        self.input_dim = input_dim
        self.embed_size = embed_size
        self.central_emb = ScaledEmbedding(self.input_dim,self.embed_size)
        self.context_emb = ScaledEmbedding(self.input_dim,self.embed_size)
        
    def forward(self, icent, icont):
        #
        # 你的代码（提示：维度分别是 icent (bs,1)、icont (bs,max_len)，输出 (bs,1,max_len)）
        #
        cent_emb = self.central_emb(icent)
        cont_emb = self.context_emb(icont)
        #print(cent_emb.shape,cont_emb.shape)
        return torch.einsum('bij,bkj -> bik' , cent_emb, cont_emb)

In [30]:
net = Skip_gram(len(idx_to_token))

In [31]:
net(torch.tensor([0,1,3]).unsqueeze(1),torch.tensor([[0,0],[0,0],[0,0]]))

tensor([[[-1.2160e-03, -1.2160e-03]],

        [[ 8.1581e-05,  8.1581e-05]],

        [[-3.7141e-04, -3.7141e-04]]], grad_fn=<ViewBackward0>)

### 损失函数

值得一提的是，我们需要用掩码变量指定小批次中参与损失函数计算的部分预测值和标签：当掩码为 1 时，相应位置的预测值和标签将参与损失函数的计算；当掩码为 0 时，相应位置的预测值和标签不参与损失函数的计算。正如我们前面提到的，掩码变量可以用来避免填充对损失函数计算的影响。


In [32]:
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
def criterion(pred, label, mask):
    #
    # 你的代码
    #
    return (loss_fn(pred, label)*mask).sum(1)/mask.sum(1)

In [33]:
pred = torch.tensor([[1.5, 0.3, -1, 2], [1.1, -0.6, 2.2, 0.4]])
# 标签变量 label 中的 1 和 0 分别表示上下文词和噪声词
# 
label = torch.tensor([[1, 0, 0, 0], [1, 1, 0, 0]]).type(torch.FloatTensor)
mask = torch.tensor([[1, 1, 1, 1], [1, 1, 1, 0]]).type(torch.FloatTensor)  # mask = torch.tensor([[1, 1, 1, 1], [1, 1, 1, 0]]).type(torch.FloatTensor)  # mask = torch.tensor([[1, 1, 1, 1], [1, 1, 1, 0]]).type(torch.FloatTensor)  # 掩码变量

criterion(pred,label,mask)

tensor([0.8740, 1.2100])

In [34]:
optimizer = torch.optim.Adam(net.parameters(),lr=0.005)

In [35]:
def train(n_epochs):
    
    for epoch in range(n_epochs):
        start, loss = time.time(), 0
        for batch in data_loader:
            #
            # 你的代码
            #
            cent, cont, mas, lab = batch
            cent = cent.to(device)
            cont = cont.to(device)
            mas = mas.to(device)
            lab = lab.type(torch.FloatTensor).to(device)
            pred = net(cent,cont).squeeze()
            optimizer.zero_grad()
            curr_loss = criterion(pred,lab,mas).mean()
            curr_loss.backward()
            optimizer.step()
            loss += curr_loss.item()
            
        print('epoch %d, loss %.2f, time %.2fs'
              % (epoch + 1, loss, time.time() - start))

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = net.to(device)

In [37]:
train(6)

epoch 1, loss 334.01, time 3.60s
epoch 2, loss 289.92, time 3.12s
epoch 3, loss 259.25, time 3.06s
epoch 4, loss 237.38, time 3.12s
epoch 5, loss 224.01, time 3.13s
epoch 6, loss 215.30, time 3.09s


## 应用词嵌入模型

训练完词嵌入模型后，我们可以根据两个词向量的余弦相似度来表示词之间的语义相似度。可以看到，使用训练好的词嵌入模型时，与 "chip" 语义最接近的词大多与芯片相关。


In [38]:
def get_similar_tokens(query_token, k, W):
    #
    # 你的代码
    #
    x = W[token_to_idx[query_token]]
    cos = torch.matmul(W,x) / torch.sqrt(torch.sum(W*W,1)*torch.sum(x*x)+1e-9)
    _,topk = torch.topk(cos, k=k+1,)
    for i in topk[1:]:# Remove the input words
        print('cosine sim=%.3f: %s' % (cos[i], (idx_to_token[i])))

get_similar_tokens('chip', 3, net.central_emb.weight.data)

cosine sim=0.549: bugs
cosine sim=0.480: intel
cosine sim=0.473: microprocessor


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)